# Consecutive-miss burst analysis

In [1]:
import pandas as pd, glob
from pathlib import Path

OUT = Path("results"); OUT.mkdir(exist_ok=True)

LABELS = {"01": "isolcpus removed", "02": "nohz_full removed", "03": "rcu_nocbs removed",
          "04": "systemd containment removed", "05": "IRQ steering removed",
          "06": "boot-params bundle removed", "08": "frequency pinning removed"}
SCENARIO_ORDER = list(LABELS)
BASELINE_LABEL = "baseline (full isolation)"
SMT_LABEL = "SMT enabled"

frames = []
for s in SCENARIO_ORDER:
    for w in ["model1", "model5"]:
        d = pd.read_csv(f"data/{s}-{w}.csv")
        d["label"] = LABELS[s]
        frames.append(d)
all_df = pd.concat(frames, ignore_index=True)

bl = []
for w, d_ in {"model1": "model1", "model5": "model5-phys", "model5-smt": "model5-smt"}.items():
    d = pd.read_csv(f"../../kubedeadline-experiments/analysis/data/{d_}.csv")
    d["workload"] = w
    d["label"] = SMT_LABEL if w == "model5-smt" else BASELINE_LABEL
    bl.append(d)
baseline_df = pd.concat(bl, ignore_index=True)

full = pd.concat([all_df, baseline_df], ignore_index=True)
print(len(full), "rows")

7300000 rows


## Longest miss run + burst share per cell

In [2]:
def burst_stats(g, min_burst_len=2):
    """g sorted by job order; deadline_miss is 0/1. A burst = a run of
    >= min_burst_len consecutive misses."""
    g = g.sort_values("job_index")
    miss = g["deadline_miss"].to_numpy()
    n_miss = int(miss.sum())
    if n_miss == 0:
        return {"n_miss": 0, "longest_run": 0, "n_bursts": 0, "misses_in_bursts": 0, "burst_share": float("nan")}

    # run-length encode consecutive misses
    runs = []
    cur = 0
    for v in miss:
        if v:
            cur += 1
        else:
            if cur:
                runs.append(cur)
            cur = 0
    if cur:
        runs.append(cur)

    longest_run = max(runs)
    bursts = [r for r in runs if r >= min_burst_len]
    misses_in_bursts = sum(bursts)
    return {
        "n_miss": n_miss,
        "longest_run": longest_run,
        "n_bursts": len(bursts),
        "misses_in_bursts": misses_in_bursts,
        "burst_share": misses_in_bursts / n_miss,
    }

rows = []
group_cols = ["label", "workload", "scale", "U", "round"]
for key, g in full.groupby(group_cols):
    row = dict(zip(group_cols, key))
    row.update(burst_stats(g))
    rows.append(row)

burst_df = pd.DataFrame(rows).sort_values(["workload", "scale", "U", "label", "round"])
burst_df.to_csv(OUT / "burst_stats_per_round.csv", index=False)
burst_df.head(20)

,label,workload,scale,U,round,n_miss,longest_run,n_bursts,misses_in_bursts,burst_share
0,IRQ steering removed,model1,soft,0.1,1,0,0,0,0,NaN
1,IRQ steering removed,model1,soft,0.1,2,0,0,0,0,NaN
2,IRQ steering removed,model1,soft,0.1,3,0,0,0,0,NaN
3,IRQ steering removed,model1,soft,0.1,4,1,1,0,0,0.000000
239,baseline (full isolation),model1,soft,0.1,1,0,0,0,0,NaN
240,baseline (full isolation),model1,soft,0.1,2,0,0,0,0,NaN
241,baseline (full isolation),model1,soft,0.1,3,2,1,0,0,0.000000
242,baseline (full isolation),model1,soft,0.1,4,1,1,0,0,0.000000
398,boot-params bundle removed,model1,soft,0.1,1,0,0,0,0,NaN
399,boot-params bundle removed,model1,soft,0.1,2,0,0,0,0,NaN


## Aggregated across rounds (per cell)

In [3]:
agg = (burst_df.groupby(["label", "workload", "scale", "U"])
       .agg(n_miss=("n_miss", "sum"),
            longest_run=("longest_run", "max"),
            n_bursts=("n_bursts", "sum"),
            misses_in_bursts=("misses_in_bursts", "sum"))
       .reset_index())
agg["burst_share"] = agg["misses_in_bursts"] / agg["n_miss"]
agg.to_csv(OUT / "burst_stats_agg.csv", index=False)
agg

,label,workload,scale,U,n_miss,longest_run,n_bursts,misses_in_bursts,burst_share
0,IRQ steering removed,model1,soft,0.10,1,1,0,0,0.000000
1,IRQ steering removed,model1,soft,0.20,0,0,0,0,NaN
2,IRQ steering removed,model1,soft,0.30,1,1,0,0,0.000000
3,IRQ steering removed,model1,soft,0.40,1,1,0,0,0.000000
4,IRQ steering removed,model1,soft,0.50,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...
321,systemd containment removed,model5,tight,0.60,14,3,1,3,0.214286
322,systemd containment removed,model5,tight,0.70,4,1,0,0,0.000000
323,systemd containment removed,model5,tight,0.80,52,4,5,13,0.250000
324,systemd containment removed,model5,tight,0.90,54,3,5,12,0.222222
